In [1]:
import pandas as pd
import psycopg2

In [2]:
## EXTRAINDO DADOS DE ARQUIVO EXTERNO
caminho_desafio_dw = 'C:\\Users\\PC\\Documents\\Desafio_dw_eletronicos\\dados\\desafio2_vendas.csv'
df_desafio_dw_vendas = pd.read_csv(caminho_desafio_dw, sep=',')
df_desafio_dw_vendas.head()

,pedido_id,data_pedido,cliente_id,produto_id,produto,categoria,marca,quantidade,preco_unitario,desconto,frete,valor_total,canal_venda,forma_pagamento,cidade,estado,status_pedido
0,1,2025-02-28,1151,102,Notebook Air 13,Notebook,TechMax,1,3643.55,109.31,1.15,3535.39,App,Boleto,Salvador,BA,Cancelado
1,2,2025-07-07,1024,106,Monitor 27 QHD,Monitores,ViewTop,1,1492.15,0.00,2.33,1494.48,App,Cartão de Crédito,Florianópolis,SC,Faturado
2,3,2025-02-24,1195,105,Monitor 24 Full HD,Monitores,ViewTop,1,945.54,75.64,9.96,879.86,Marketplace,Pix,Campinas,SP,Faturado
3,4,2025-05-11,1182,111,Tablet T10,Tablet,MobiOne,1,1294.88,38.85,29.56,1285.59,Site,Boleto,Porto Alegre,RS,Entregue
4,5,2025-03-30,1374,105,Monitor 24 Full HD,Monitores,ViewTop,1,1000.43,0.00,3.25,1003.68,Site,Cartão de Crédito,Campinas,SP,Cancelado


In [3]:
#Verificar informações da tabela
df_desafio_dw_vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pedido_id        3000 non-null   int64  
 1   data_pedido      3000 non-null   str    
 2   cliente_id       3000 non-null   int64  
 3   produto_id       3000 non-null   int64  
 4   produto          3000 non-null   str    
 5   categoria        3000 non-null   str    
 6   marca            3000 non-null   str    
 7   quantidade       3000 non-null   int64  
 8   preco_unitario   3000 non-null   float64
 9   desconto         3000 non-null   float64
 10  frete            3000 non-null   float64
 11  valor_total      3000 non-null   float64
 12  canal_venda      3000 non-null   str    
 13  forma_pagamento  3000 non-null   str    
 14  cidade           3000 non-null   str    
 15  estado           3000 non-null   str    
 16  status_pedido    3000 non-null   str    
dtypes: float64(4), int64(4), 

In [4]:
#Caso tenha valores ausentes será tratado para "sem informacao"
df_desafio_dw_vendas['estado'] = df_desafio_dw_vendas['estado'].fillna('Sem Informação')
df_desafio_dw_vendas['forma_pagamento'] = df_desafio_dw_vendas['forma_pagamento'].fillna('Sem Informação')

df_desafio_dw_vendas.head()

,pedido_id,data_pedido,cliente_id,produto_id,produto,categoria,marca,quantidade,preco_unitario,desconto,frete,valor_total,canal_venda,forma_pagamento,cidade,estado,status_pedido
0,1,2025-02-28,1151,102,Notebook Air 13,Notebook,TechMax,1,3643.55,109.31,1.15,3535.39,App,Boleto,Salvador,BA,Cancelado
1,2,2025-07-07,1024,106,Monitor 27 QHD,Monitores,ViewTop,1,1492.15,0.00,2.33,1494.48,App,Cartão de Crédito,Florianópolis,SC,Faturado
2,3,2025-02-24,1195,105,Monitor 24 Full HD,Monitores,ViewTop,1,945.54,75.64,9.96,879.86,Marketplace,Pix,Campinas,SP,Faturado
3,4,2025-05-11,1182,111,Tablet T10,Tablet,MobiOne,1,1294.88,38.85,29.56,1285.59,Site,Boleto,Porto Alegre,RS,Entregue
4,5,2025-03-30,1374,105,Monitor 24 Full HD,Monitores,ViewTop,1,1000.43,0.00,3.25,1003.68,Site,Cartão de Crédito,Campinas,SP,Cancelado


In [5]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

## CRIANDO TABELA raw_vendas NO BANCO DE DADOS
cursor.execute(""" 
               
                CREATE TABLE IF NOT EXISTS raw.raw_vendas
                    (
                        pedido_id integer,
                        data_pedido date,
                        cliente_id integer,
                        produto_id integer,
                        produto varchar(100),
                        categoria varchar(100),
                        marca varchar(100),
                        quantidade integer,
                        preco_unitario decimal(10,2),
                        desconto decimal(10,2),
                        frete decimal(10,2),
                        valor_total decimal(10,2),
                        canal_venda varchar(50),
                        forma_pagamento varchar(50),
                        cidade varchar(50),
                        estado varchar(5),
                        status_pedido varchar(50)
                    );

               """)
conexao.commit()
cursor.close()
conexao.close()

In [6]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()


cursor.execute('delete from raw.raw_vendas')

##SUBINDO DADOS DA TABELA EXTRAIDA PARA O BANCO DE DADOS 
for i, df_desafio_dw_vendas_raw in df_desafio_dw_vendas.iterrows():
    cursor.execute(""" insert into raw.raw_vendas (
                   pedido_id,
                   data_pedido,
                   cliente_id,
                   produto_id,
                   produto,
                   categoria,
                   marca,
                   quantidade,
                   preco_unitario,
                   desconto,
                   frete,
                   valor_total,
                   canal_venda,
                   forma_pagamento,
                   cidade,
                   estado,
                   status_pedido
                   ) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                   """, (
                       df_desafio_dw_vendas_raw['pedido_id'],
                       df_desafio_dw_vendas_raw['data_pedido'],
                       df_desafio_dw_vendas_raw['cliente_id'],
                       df_desafio_dw_vendas_raw['produto_id'],
                       df_desafio_dw_vendas_raw['produto'],
                       df_desafio_dw_vendas_raw['categoria'],
                       df_desafio_dw_vendas_raw['marca'],
                       df_desafio_dw_vendas_raw['quantidade'],
                       df_desafio_dw_vendas_raw['preco_unitario'],
                       df_desafio_dw_vendas_raw['desconto'],
                       df_desafio_dw_vendas_raw['frete'],
                       df_desafio_dw_vendas_raw['valor_total'],
                       df_desafio_dw_vendas_raw['canal_venda'],
                       df_desafio_dw_vendas_raw['forma_pagamento'],
                       df_desafio_dw_vendas_raw['cidade'],
                       df_desafio_dw_vendas_raw['estado'],
                       df_desafio_dw_vendas_raw['status_pedido']
                   )
                   )
conexao.commit()
cursor.close()
conexao.close()

In [7]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

                    CREATE TABLE IF NOT EXISTS staging.stg_vendas AS 
                            SELECT DISTINCT 
                                pedido_id,
                                data_pedido,
                                cliente_id,
                                produto_id,
                                TRIM(produto) AS produto,
                                TRIM(categoria) AS categoria_produto,
                                LOWER(marca) AS marca_produto,
                                quantidade,
                                preco_unitario::NUMERIC(10,2),
                                desconto::NUMERIC(10,2),
                                frete::NUMERIC(10,2),
                                valor_total::NUMERIC(10,2),
                                LOWER(canal_venda) AS canal_vendas,
                                LOWER(forma_pagamento) AS forma_pagamento,
                                TRIM(cidade) AS cidade,
                                UPPER(estado) AS estado,
                                UPPER(status_pedido) AS status_pedido,

                                CASE 
                                    WHEN UPPER(status_pedido) = 'CANCELADO' THEN 0
                                    ELSE (
                                        valor_total::NUMERIC(10,2)
                                        - desconto::NUMERIC(10,2)
                                        - frete::NUMERIC(10,2)
                                    )
                                END AS faturamento_liquido

                            FROM raw.raw_vendas
                            WHERE preco_unitario IS NOT NULL;
                                   

               """)
conexao.commit()
cursor.close()
conexao.close()
